<a href="https://colab.research.google.com/github/yukyoungs/hr-feedback-sentiment-analysis/blob/main/hr_feedback_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pandas

In [ ]:
import pandas as pd

data = {
    "id": range(1, 11),
    "employee_id": ["E001","E002","E003","E004","E005",
                    "E006","E007","E008","E009","E010"],
    "feedback": [
        "업무 환경이 매우 좋고 팀워크가 훌륭합니다.",
        "야근이 너무 많아서 힘들어요.",
        "복지 제도가 잘 갖춰져 있어 만족스럽습니다.",
        "상사와의 소통이 부족한 것 같습니다.",
        "성장 기회가 많아서 동기부여가 됩니다.",
        "급여 수준이 기대에 미치지 못합니다.",
        "자율적인 근무 문화가 마음에 듭니다.",
        "업무 프로세스가 비효율적입니다.",
        "동료들이 친절하고 협력적입니다.",
        "업무량이 너무 많아 번아웃이 걱정됩니다.",
    ]
}

df = pd.DataFrame(data)
df.to_csv("hr_feedback.csv", index=False)
print(df)


In [ ]:
from transformers import pipeline

print("모델 다운로드 중...")

classifier = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

df = pd.read_csv("hr_feedback.csv")
results = classifier(df["feedback"].tolist())

df["sentiment"] = [r["label"] for r in results]
df["score"] = [round(r["score"], 4) for r in results]

print(df[["employee_id", "feedback", "sentiment", "score"]])


In [ ]:
import sqlite3

conn = sqlite3.connect("hr_analysis.db")
df.to_sql("feedback_analysis", conn, if_exists="replace", index=False)

# 저장 확인
result = pd.read_sql("SELECT * FROM feedback_analysis", conn)
print("=== DB 저장 완료 ===")
print(result)

# 감성별 통계
stats = pd.read_sql("""
    SELECT sentiment, COUNT(*) as count
    FROM feedback_analysis
    GROUP BY sentiment
""", conn)
print("\n=== 감성 분석 요약 ===")
print(stats)

conn.close()
